In [5]:
import pandas as pd

from src.thetadata_pipeline.ib.lattency import trades_collector
from src.thetadata_pipeline.pipeline_config import load_pipeline_config
from src.thetadata_pipeline.settings import get_settings

cfg = load_pipeline_config()
settings = get_settings()

In [10]:
strangle_trades = pd.read_parquet(get_settings().strategies_dir / 'SPY_strangle_trades.parquet')

ib_files = list(settings.ib_states_dir.glob(f"{cfg.analysis.account_id}*.csv"))
ib_trades = trades_collector(cfg.analysis.account_id, settings)

In [13]:
ib_trades = ib_trades.drop(
    columns=['Trades', 'Header', 'DataDiscriminator', 'Asset Category', 'Currency', 'C. Price', 'Basis',
             'MTM P/L', 'Code', 'code', 'Quantity']
)

In [17]:
ib_trades.sort_values('trade_dt')

,Symbol,Date/Time,T. Price,Proceeds,Comm/Fee,Realized P/L,ticker,expiration,strike,right,trade_dt,trade_date,trade_ms,quantity
0,AAPL 22NOV24 227.5 C,"2024-11-19, 14:34:02",2.98,298,-1.0626244,0,AAPL,2024-11-22,227.5,C,2024-11-19 14:34:02,2024-11-19,52442000,-1
2,AAPL 22NOV24 227.5 P,"2024-11-19, 14:34:02",1.22,-122,-1.05155,0,AAPL,2024-11-22,227.5,P,2024-11-19 14:34:02,2024-11-19,52442000,1
1,AAPL 22NOV24 227.5 C,"2024-11-22, 16:20:00",0,0,0,0,AAPL,2024-11-22,227.5,C,2024-11-22 16:20:00,2024-11-22,58800000,1
3,AAPL 22NOV24 227.5 P,"2024-11-22, 16:20:00",0,0,0,-123.05155,AAPL,2024-11-22,227.5,P,2024-11-22 16:20:00,2024-11-22,58800000,-1
4,SPY 18DEC24 605 C,"2024-12-18, 09:37:55",1.07,107,-1.5397146,0,SPY,2024-12-18,605.0,C,2024-12-18 09:37:55,2024-12-18,34675000,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1489,SPY 15MAY26 742 C,"2026-05-15, 10:18:46",1.94,-776,-2.753,-294.889213,SPY,2026-05-15,742.0,C,2026-05-15 10:18:46,2026-05-15,37126000,4
1494,SPY 18MAY26 737 P,"2026-05-18, 09:36:04",1.16,464,-2.0557184,0,SPY,2026-05-18,737.0,P,2026-05-18 09:36:04,2026-05-18,34564000,-4
1492,SPY 18MAY26 740 C,"2026-05-18, 09:36:04",1.19,476,-2.8159656,0,SPY,2026-05-18,740.0,C,2026-05-18 09:36:04,2026-05-18,34564000,-4
1493,SPY 18MAY26 740 C,"2026-05-18, 09:51:34",1.65,-660,-2.793,-189.608964,SPY,2026-05-18,740.0,C,2026-05-18 09:51:34,2026-05-18,35494000,4


In [ ]:
# important_cols = [
#     'Symbol', 'Date/Time', 'T. Price', 'Proceeds', 'Comm/Fee', 'Realized P/L',
#     'ticker', 'expiration', 'strike', 'right', 'trade_dt', 'trade_date', 'trade_ms', 'quantity'
# ]
# ib_trades = ib_trades[important_cols].copy()

for col in ['T. Price', 'Proceeds', 'Comm/Fee', 'Realized P/L', 'strike', 'trade_ms', 'quantity']:
    ib_trades[col] = pd.to_numeric(ib_trades[col], errors='coerce')
ib_trades['right'] = ib_trades['right'].str.lower()

leg_keys = ['ticker', 'expiration', 'strike', 'right']

def summarize_side(rows: pd.DataFrame, prefix: str) -> pd.DataFrame:
    return (
        rows.sort_values('trade_dt')
        .groupby(leg_keys, as_index=False)
        .agg(
            **{
                f'{prefix}_Symbol': ('Symbol', 'first'),
                f'{prefix}_Date/Time': ('Date/Time', 'first'),
                f'{prefix}_fill_price': ('T. Price', 'first'),
                f'{prefix}_proceeds': ('Proceeds', 'sum'),
                f'{prefix}_comm_fee': ('Comm/Fee', 'sum'),
                f'{prefix}_realized_pl': ('Realized P/L', 'sum'),
                f'{prefix}_trade_dt': ('trade_dt', 'first'),
                f'{prefix}_trade_date': ('trade_date', 'first'),
                f'{prefix}_trade_ms': ('trade_ms', 'first'),
                f'{prefix}_quantity': ('quantity', 'sum'),
            }
        )
    )

# Short option logic: sell/open rows are negative quantity, buy/expire/close rows are positive quantity.
entries = summarize_side(ib_trades[ib_trades['quantity'] < 0], 'ent')
exits = summarize_side(ib_trades[ib_trades['quantity'] > 0], 'ext')

ib_legs = entries.merge(exits, on=leg_keys, how='left')
ib_legs['date'] = ib_legs['ent_trade_date']
ib_legs['contracts'] = ib_legs['ent_quantity'].abs()
ib_legs['ent_opt_bid'] = ib_legs['ent_proceeds'] / ib_legs['contracts'] / 100
ib_legs['ext_opt_ask'] = (-ib_legs['ext_proceeds'].fillna(0)) / ib_legs['contracts'] / 100
ib_legs['leg_comm_fee'] = ib_legs['ent_comm_fee'].fillna(0) + ib_legs['ext_comm_fee'].fillna(0)
ib_legs['leg_profit_cash'] = ib_legs['ent_proceeds'].fillna(0) + ib_legs['ext_proceeds'].fillna(0) + ib_legs['leg_comm_fee']
ib_legs['leg_profit'] = ib_legs['leg_profit_cash'] / ib_legs['contracts'] / 100

join_cols = ['ticker', 'expiration', 'date']
call = ib_legs[ib_legs['right'] == 'c'].drop(columns='right')
put = ib_legs[ib_legs['right'] == 'p'].drop(columns='right')
call = call.rename(columns={c: f'call_{c}' for c in call.columns if c not in join_cols})
put = put.rename(columns={c: f'put_{c}' for c in put.columns if c not in join_cols})

ib_strangles = call.merge(put, on=join_cols, how='inner')
ib_strangles['ib_strangle_profit_cash'] = ib_strangles['call_leg_profit_cash'] + ib_strangles['put_leg_profit_cash']
ib_strangles['ib_strangle_profit'] = ib_strangles['call_leg_profit'] + ib_strangles['put_leg_profit']
ib_strangles['ib_comm_fee'] = ib_strangles['call_leg_comm_fee'] + ib_strangles['put_leg_comm_fee']

compare_df = strangle_trades.merge(
    ib_strangles,
    on=['ticker', 'expiration', 'date'],
    how='inner',
    suffixes=('_bt', '_ib'),
)

ib_strangles.head()
